<a href="https://colab.research.google.com/github/nvreetk280906-eng/TorchONNX/blob/main/housepriceregression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install onnx onnxruntime skl2onnx onnxscript

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 42.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 36.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.2/317.2 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 689.1/689.1 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.1/164.1 kB 11.2 MB/s eta 0:00:00


In [2]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from torch.optim import Adam
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
import onnx
import onnxruntime as ort
print('All libraries loaded!')

All libraries loaded!


In [3]:
BATCH_SIZE   = 32
LEARNING_RATE = 0.001
EPOCHS       = 150
RANDOM_SEED  = 42
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

In [4]:
housing = fetch_california_housing(as_frame=True)
df = housing.frame
print('Dataset shape:', df.shape)
print('\nFeature names:', housing.feature_names)
print('\nTarget: MedHouseVal (median house value in $100k)')
df.head()

Dataset shape: (20640, 9)

Feature names: ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude']

Target: MedHouseVal (median house value in $100k)


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422


In [5]:
df.describe()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
count,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000
mean,3.870671,28.639486,5.429000,1.096675,1425.476744,3.070655,35.631861,-119.569704,2.068558
std,1.899822,12.585558,2.474173,0.473911,1132.462122,10.386050,2.135952,2.003532,1.153956
min,0.499900,1.000000,0.846154,0.333333,3.000000,0.692308,32.540000,-124.350000,0.149990
25%,2.563400,18.000000,4.440716,1.006079,787.000000,2.429741,33.930000,-121.800000,1.196000
50%,3.534800,29.000000,5.229129,1.048780,1166.000000,2.818116,34.260000,-118.490000,1.797000
75%,4.743250,37.000000,6.052381,1.099526,1725.000000,3.282261,37.710000,-118.010000,2.647250
max,15.000100,52.000000,141.909091,34.066667,35682.000000,1243.333333,41.950000,-114.310000,5.000010


In [6]:
# Separate features and target
X = df[housing.feature_names].values   # shape: (20640, 8)
y = df['MedHouseVal'].values            # shape: (20640,)

print('X shape:', X.shape)
print('y shape:', y.shape)
print('y range: [{:.2f}, {:.2f}]'.format(y.min(), y.max()))

X shape: (20640, 8)
y shape: (20640,)
y range: [0.15, 5.00]


In [7]:
# Train/test split — 80/20
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED
)
print('Train:', X_train.shape, '| Test:', X_test.shape)

Train: (16512, 8) | Test: (4128, 8)


In [8]:
# Convert to float32
X_train = X_train.astype(np.float32)
X_test  = X_test.astype(np.float32)
y_train = y_train.astype(np.float32)
y_test  = y_test.astype(np.float32)

# Compute mean & std from TRAINING set only (to avoid data leakage)
X_means = X_train.mean(axis=0)
X_stds  = X_train.std(axis=0) + 1e-8   # epsilon to avoid division by zero

print('Feature means:', np.round(X_means, 3))
print('Feature stds: ', np.round(X_stds,  3))

Feature means: [ 3.881000e+00  2.860800e+01  5.435000e+00  1.097000e+00  1.426451e+03
  3.097000e+00  3.564300e+01 -1.195820e+02]
Feature stds:  [1.904000e+00 1.260200e+01 2.387000e+00 4.330000e-01 1.137022e+03
 1.157800e+01 2.137000e+00 2.006000e+00]


In [9]:
# Convert to PyTorch tensors
X_train_t = torch.tensor(X_train)
X_test_t  = torch.tensor(X_test)
y_train_t = torch.tensor(y_train).unsqueeze(1)  # shape: (N, 1)
y_test_t  = torch.tensor(y_test).unsqueeze(1)

print('X_train_t:', X_train_t.shape)
print('y_train_t:', y_train_t.shape)

X_train_t: torch.Size([16512, 8])
y_train_t: torch.Size([16512, 1])


In [10]:
# Build DataLoader using TensorDataset
train_ds     = TensorDataset(X_train_t, y_train_t)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
print('DataLoader ready — batches per epoch:', len(train_loader))

DataLoader ready — batches per epoch: 516


In [11]:
class HousePriceDLNet(nn.Module):
    def __init__(self, x_means, x_stds):
        super(HousePriceDLNet, self).__init__()

        # Store normalization stats as non-trainable buffers
        self.register_buffer('x_means', torch.tensor(x_means, dtype=torch.float32))
        self.register_buffer('x_stds',  torch.tensor(x_stds,  dtype=torch.float32))

        # Architecture: 8 -> 64 -> 32 -> 16 -> 1
        self.linear1 = nn.Linear(8,  64)
        self.act1    = nn.ReLU()
        self.drop1   = nn.Dropout(0.2)

        self.linear2 = nn.Linear(64, 32)
        self.act2    = nn.ReLU()
        self.drop2   = nn.Dropout(0.2)

        self.linear3 = nn.Linear(32, 16)
        self.act3    = nn.ReLU()

        self.linear4 = nn.Linear(16, 1)   # output layer — no activation (regression)

    def forward(self, x):
        # Standardize input
        x = (x - self.x_means) / self.x_stds

        # Forward pass
        x = self.drop1(self.act1(self.linear1(x)))
        x = self.drop2(self.act2(self.linear2(x)))
        x = self.act3(self.linear3(x))
        x = self.linear4(x)
        return x


model = HousePriceDLNet(X_means, X_stds)
print(model)

HousePriceDLNet(
  (linear1): Linear(in_features=8, out_features=64, bias=True)
  (act1): ReLU()
  (drop1): Dropout(p=0.2, inplace=False)
  (linear2): Linear(in_features=64, out_features=32, bias=True)
  (act2): ReLU()
  (drop2): Dropout(p=0.2, inplace=False)
  (linear3): Linear(in_features=32, out_features=16, bias=True)
  (act3): ReLU()
  (linear4): Linear(in_features=16, out_features=1, bias=True)
)


In [12]:
optimizer = Adam(model.parameters(), lr=LEARNING_RATE)
loss_fn   = nn.MSELoss()   # MSE for regression (not cross-entropy)

loss_history = []

for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0.0

    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        y_pred = model(X_batch)
        loss   = loss_fn(y_pred, y_batch)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(train_loader)
    loss_history.append(avg_loss)

    if (epoch + 1) % 25 == 0:
        print(f'Epoch [{epoch+1:3d}/{EPOCHS}]  MSE Loss: {avg_loss:.4f}')

print('\nTraining complete!')

Epoch [ 25/150]  MSE Loss: 0.3184
Epoch [ 50/150]  MSE Loss: 0.2879
Epoch [ 75/150]  MSE Loss: 0.2760
Epoch [100/150]  MSE Loss: 0.2760
Epoch [125/150]  MSE Loss: 0.2687
Epoch [150/150]  MSE Loss: 0.2626

Training complete!


In [13]:
model.eval()
with torch.no_grad():
    y_pred_test = model(X_test_t)

y_pred_np = y_pred_test.detach().numpy()
y_test_np = y_test_t.numpy()

r2 = r2_score(y_test_np, y_pred_np)
print(f'R² Score on Test Set: {r2:.4f}')

# Show a few sample predictions
print('\nSample Predictions (in $100k):')
print(f'{"Predicted":>12}  {"Actual":>8}')
for i in range(10):
    print(f'{y_pred_np[i][0]:>12.2f}  {y_test_np[i][0]:>8.2f}')

R² Score on Test Set: 0.7849

Sample Predictions (in $100k):
   Predicted    Actual
        0.74      0.48
        1.12      0.46
        4.49      5.00
        2.27      2.19
        2.23      2.78
        1.57      1.59
        2.15      1.98
        1.67      1.58
        2.29      3.40
        4.51      4.47


In [14]:
model.eval()

# Dummy input: 1 sample, 8 features
dummy_input = torch.zeros(1, 8, dtype=torch.float32)

onnx_filename = 'house_price_model.onnx'

torch.onnx.export(
    model,
    dummy_input,
    onnx_filename,
    export_params   = True,
    opset_version   = 15,          # must match the JS ONNX runtime version
    do_constant_folding = True,
    input_names     = ['input'],
    output_names    = ['output'],
    dynamic_axes    = {'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}},
    dynamo          = False
)

print(f'Model exported to: {onnx_filename}')

# Verify the exported model
onnx_model = onnx.load(onnx_filename)
onnx.checker.check_model(onnx_model)
print('ONNX model verified successfully!')

Model exported to: house_price_model.onnx
ONNX model verified successfully!


/tmp/ipykernel_4439/1301677857.py:8: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


In [15]:
# Quick sanity check: run the ONNX model locally
sess = ort.InferenceSession(onnx_filename)
sample = X_test[:1].astype(np.float32)
onnx_out = sess.run(['output'], {'input': sample})
torch_out = model(torch.tensor(sample)).item()

print(f'ONNX output:  {onnx_out[0][0][0]:.4f}')
print(f'PyTorch output: {torch_out:.4f}')
print(f'Actual value:   {y_test[0]:.4f}')
print('\nONNX output matches PyTorch — ready for deployment!')

ONNX output:  0.7423
PyTorch output: 0.7423
Actual value:   0.4770

ONNX output matches PyTorch — ready for deployment!


In [16]:
try:
    from google.colab import files
    files.download('house_price_model.onnx')
    print('Download started!')
except ImportError:
    print('Not running in Colab — find house_price_model.onnx in your working directory.')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download started!
